# 04 · Build the final dataset, leakage checks, and splits
**Purpose:** turn validated human paragraphs + raw generations into one balanced, topic-paired, leakage-checked dataset with deterministic splits; prepare the external COLING-kk test set; audit surface artefacts.

**Leakage controls:**
1. identical normalisation for both classes (no formatting shortcuts);
2. exact + MinHash near-duplicate removal within each language, and against the COLING-kk external set;
3. **split by Wikidata QID hash** — a topic is in the same split in every language, so cross-lingual tests never see a training topic;
4. **pair-level length matching** (v1.1): the longer text of each human/machine pair is cut at the sentence boundary closest to the shorter text's length, so length cannot act as a shortcut (it did: length-only AUROC 0.90 in en, 0.38 in ky before matching);
5. balanced cells: same N per (language × generator), equal human/machine.

**v1.2 (2026-10-10):** human texts are re-normalised here with the v1.2 `normalize_text` (parenthetical/bracketed spans removed from both classes, stress marks, italics, trailing LLM notes), so 02/03 need not be re-run; machine texts with reader-addressing meta-commentary ("Would you like more details…", "Егер сізде … болса") are dropped like refusals; the audit adds a stylometric baseline and a per-character shortcut scan.

**Outputs:** `03_processed/dataset_v1.parquet`, `03_processed/external_v1.parquet`, `04_splits/*.json|csv`, `08_results/data_audit.csv`.
**Runtime:** CPU, a few minutes.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "04_build_dataset_and_splits"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
!pip -q install datasketch fasttext

In [ ]:
from datasketch import MinHash, MinHashLSH
migrate_config(); D = CFG["data"]
per_cell = D["n_topics_target"] // len(GENERATORS)

## 1 · Load and normalise machine texts; pair with human texts

In [ ]:
REFUSAL = re.compile(r"(?:I'm sorry|I cannot|I can't|As an AI|as a language model|Извините|Кешіріңіз|Üzgünüm)", re.I)
# reader-addressing meta-commentary (trivial machine cue + low-quality sample) -> dropped, counted in meta_dropped.csv
META = re.compile(r"(?:Would you like|If you (?:have|need|want|would like)|please (?:share|let me know|provide)|"
                  r"I (?:don't|do not|couldn't|could not) (?:have|find)|knowledge cut-?off|as of my last|"
                  r"Если у вас есть|Если вы хотите|Хотите ли вы|Егер сізде|Егер сіз |Эгер сизде|Эгер сиз |"
                  r"Agar sizda|Agar siz |isterseniz|bilginiz varsa)", re.I)
meta_log = []
pairs = []
for lang in LANGS:
    hum = pd.read_parquet(P["validated"] / f"human_{lang}.parquet")
    hum["text"] = hum.text.map(lambda t: truncate_words(normalize_text(t), D["max_words"]))     # v1.2 re-normalisation
    hum = hum[hum.text.map(n_words) >= D["min_words"]]
    gen = pd.DataFrame(read_jsonl(P["raw_gen"] / f"{lang}.jsonl"))
    gen = gen[gen.text.notna()].drop_duplicates("key").drop_duplicates(["lang", "title"])   # one generation per topic
    gen["m_text"] = gen.text.map(lambda t: truncate_words(normalize_text(t), D["max_words"]))
    is_meta = gen.m_text.str.contains(META)
    meta_log += [{"lang": lang, "generator": g, "n_meta": int(n)} for g, n in gen[is_meta].groupby("generator").size().items()]
    gen = gen[(gen.m_text.map(n_words) >= D["min_words"]) & (~gen.m_text.str.contains(REFUSAL)) & (~is_meta)]
    lid = langid(gen.m_text.tolist())
    gen = gen[[(l == LANG_META[lang]["lid"]) and (p >= D["langid_min_prob"]) for l, p in lid]]
    m = gen.merge(hum[["title", "qid", "text", "revid", "rev_ts"]].rename(columns={"text": "h_text", "qid": "h_qid"}), on="title", how="inner")
    m["qid"] = m.h_qid.fillna(m.qid)
    pairs.append(m[["lang", "title", "qid", "pair_id", "generator", "family", "model", "served_model", "provider",
                    "prompt_id", "topic_rank", "revid", "rev_ts", "h_text", "m_text"]])
    log.info(f"[{lang}] valid pairs: {len(m)}")
pairs = pd.concat(pairs, ignore_index=True)
pd.DataFrame(meta_log, columns=["lang", "generator", "n_meta"]).to_csv(P["validated"] / "meta_dropped.csv", index=False)
log.info(f"meta-commentary generations dropped: {sum(r['n_meta'] for r in meta_log)}")

## 2 · Near-duplicate removal (MinHash, word 3-shingles) — within language and vs. COLING-kk

In [ ]:
def minhash(text, num_perm=D["minhash_perm"]):
    w = text.lower().split(); mh = MinHash(num_perm=num_perm, seed=1)
    for i in range(max(1, len(w) - 2)):
        mh.update(" ".join(w[i:i + 3]).encode("utf-8"))
    return mh

coling = pd.DataFrame(read_jsonl(P["raw_external"] / "coling2025_kazakh.jsonl"))
coling["text_norm"] = coling.text.map(normalize_text)

drop = set(); dup_log = []
for lang in LANGS:
    sub = pairs[pairs.lang == lang]
    lsh = MinHashLSH(threshold=D["minhash_threshold"], num_perm=D["minhash_perm"])
    if lang == "kk":     # stricter threshold against the external test set (leakage into it would inflate external results)
        lsh_ext = MinHashLSH(threshold=0.5, num_perm=D["minhash_perm"])
        for i, t in enumerate(coling.text_norm):
            lsh_ext.insert(f"ext{i}", minhash(t))
    for idx, r in sub.sort_values("topic_rank").iterrows():
        hit = False
        for side in ("h_text", "m_text"):
            mh = minhash(r[side])
            if lsh.query(mh) or (lang == "kk" and lsh_ext.query(mh)):
                hit = True; dup_log.append({"lang": lang, "pair_id": r.pair_id, "side": side}); break
        if hit:
            drop.add(idx); continue
        lsh.insert(f"{idx}h", minhash(r.h_text)); lsh.insert(f"{idx}m", minhash(r.m_text))
pairs = pairs.drop(index=list(drop)).reset_index(drop=True)
pd.DataFrame(dup_log).to_csv(P["validated"] / "near_duplicates_removed.csv", index=False)
log.info(f"near-duplicate pairs removed: {len(drop)}")

## 2b · Pair-level length matching (sentence boundaries only)

In [ ]:
def sentences(t):
    return [x for x in _SENT_END.split(t) if x.strip()]

def match_pair(h, m):
    nh, nm = n_words(h), n_words(m)
    L = min(nh, nm)
    if nh == nm:
        return h, m, True
    longer = h if nh > nm else m
    sents = sentences(longer); cum = np.cumsum([n_words(x) for x in sents])
    j = int(np.argmin(np.abs(cum - L))); cut = " ".join(sents[:j + 1]); c = int(cum[j])
    ok = abs(c - L) <= max(D["length_tol_words"], D["length_tol_frac"] * L) and c >= D["min_words"]
    return (cut, m, ok) if nh > nm else (h, cut, ok)

res = [match_pair(h, m) for h, m in zip(pairs.h_text, pairs.m_text)]
pairs["h_text"], pairs["m_text"] = [r[0] for r in res], [r[1] for r in res]
pairs["len_ok"] = [r[2] for r in res]
len_log = pairs.groupby("lang").len_ok.agg(["size", "sum"]).rename(columns={"size": "pairs", "sum": "kept"})
len_log.to_csv(P["validated"] / "length_matching.csv"); log.info(f"length matching:\n{len_log}")
pairs = pairs[pairs.len_ok].drop(columns="len_ok").reset_index(drop=True)

## 3 · Balance cells and assign splits by QID hash

In [ ]:
cell = pairs.groupby(["lang", "generator"]).size().unstack(fill_value=0)
n_cell = int(min(cell.values.min(), per_cell))
log.info(f"pairs per (lang×generator) available:\n{cell}\n→ balanced N per cell = {n_cell}")
if n_cell < per_cell:
    log.warning("Balanced N is below target — report this in the paper (data availability limit).")
bal = (pairs.sort_values("topic_rank").groupby(["lang", "generator"], group_keys=False).head(n_cell).reset_index(drop=True))

def split_of(key):
    b = int(hashlib.sha1(key.encode("utf-8")).hexdigest(), 16) % 100
    s = D["split_pct"]
    return "train" if b < s["train"] else ("val" if b < s["train"] + s["val"] else "test")
bal["split_key"] = [q if isinstance(q, str) and q.startswith("Q") else f"{l}:{t}" for q, l, t in zip(bal.qid, bal.lang, bal.title)]
bal["split"] = bal.split_key.map(split_of)

rows = []
for r in bal.itertuples():
    base = {"lang": r.lang, "title": r.title, "qid": r.qid, "pair_id": r.pair_id, "split": r.split,
            "prompt_id": r.prompt_id, "pair_generator": r.generator, "pair_family": r.family}
    rows.append(base | {"doc_id": f"{r.pair_id}:h", "label": 0, "generator": "human", "family": "human", "text": r.h_text,
                        "source_meta": json.dumps({"revid": r.revid, "rev_ts": r.rev_ts})})
    rows.append(base | {"doc_id": f"{r.pair_id}:m", "label": 1, "generator": r.generator, "family": r.family, "text": r.m_text,
                        "source_meta": json.dumps({"model": r.model, "served_model": r.served_model, "provider": r.provider})})
ds = pd.DataFrame(rows)
ds["n_words"] = ds.text.map(n_words)
assert ds.doc_id.is_unique
# cross-lingual leakage check: no QID in test (any language) appears in train (any language)
q_train = set(ds[(ds.split == "train") & ds.qid.notna()].qid); q_test = set(ds[(ds.split == "test") & ds.qid.notna()].qid)
assert not (q_train & q_test), "QID leakage between train and test"
save_parquet(ds, P["processed"] / f"dataset_{CFG['data_version']}.parquet")
summary = ds.groupby(["lang", "split", "label"]).size().unstack([1, 2])
summary.to_csv(P["splits"] / "split_summary.csv"); log.info(f"\n{summary}")
for sp in ("train", "val", "test"):
    write_json(P["splits"] / f"{sp}_doc_ids.json", sorted(ds[ds.split == sp].doc_id))

## 4 · External test set: COLING-2025 Kazakh (GPT-4o, Wikipedia, one paragraph)
Two versions: **raw** (as distributed) and **norm** (our normalisation). Comparing them quantifies how much of any detector's external accuracy comes from formatting artefacts.

In [ ]:
ext = []
for ver in ("raw", "norm"):
    for r in coling.itertuples():
        t = r.text if ver == "raw" else r.text_norm
        lab = int(r.label)
        ext.append({"doc_id": f"colingkk-{ver}-{r.testset_id}", "orig_doc_id": f"colingkk-{r.testset_id}",
                    "eval_set": f"ext_coling_{ver}", "lang": "kk", "label": lab,
                    "family": "openai" if lab == 1 else "human", "generator": "gpt4o_coling" if lab == 1 else "human",
                    "text": t})
ext = pd.DataFrame(ext); save_parquet(ext, P["processed"] / f"external_{CFG['data_version']}.parquet")

## 5 · Surface-artefact audit (data-quality evidence for the paper)
Cross-validated AUROC (5-fold logistic regression) of shallow features only:
- **artefacts** — double spaces, space before punctuation, newlines, markdown: must be ≈ 0.5 in our data;
- **punct_bag** — per-word density of every punctuation character; **sent_len** — mean sentence length;
  **stylometric** = both. These capture genuine style differences (LLMs write longer sentences than Wikipedia leads),
  so they are *not* removed; instead the stylometric model is reported as a shallow **baseline detector** that
  neural detectors must beat (also cross-lingually);
- **length** and **title_start** (text begins with the article title).
`char_shortcuts.csv`: every non-alphanumeric character with its single-feature AUROC and document frequency per class,
to spot any remaining one-character shortcut.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PUNCT = list(".,;:!?—-\"'()[]/%…*")
def _sents(t): return [x for x in _SENT_END.split(t) if x.strip()]
def surface_feats(texts):
    nw = np.array([max(1, n_words(t)) for t in texts])
    f = pd.DataFrame({
        "double_space": [int("  " in t) for t in texts],
        "space_before_punct": [int(bool(re.search(r" [.,;:!?)]", t))) for t in texts],
        "newline": [int("\n" in t) for t in texts],
        "markdown": [int(bool(re.search(r"\*|^#", t, re.M))) for t in texts],
        "sent_len": [np.mean([n_words(x) for x in _sents(t)] or [0]) for t in texts]})
    for c in PUNCT:
        f[f"p_{c}"] = [t.count(c) for t in texts] / nw
    return f

def cv_auc(X, y):
    if np.ptp(X, axis=0).sum() == 0:
        return 0.5
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    p = cross_val_predict(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    return auroc(y, p)

ART = ["double_space", "space_before_punct", "newline", "markdown"]
BAG = [f"p_{c}" for c in PUNCT]
audit, chars = [], []
sets = [(f"ours_{l}", ds[ds.lang == l]) for l in LANGS] + [(k, ext[ext.eval_set == k]) for k in ("ext_coling_raw", "ext_coling_norm")]
for name, d in sets:
    y = d.label.values; texts = d.text.tolist(); F = surface_feats(texts)
    audit.append({"set": name, "n": len(d),
                  "auroc_artefacts_only": cv_auc(F[ART].values, y),
                  "auroc_punct_bag": cv_auc(F[BAG].values, y),
                  "auroc_sent_len": auroc(y, F.sent_len.values),
                  "auroc_stylometric": cv_auc(F[BAG + ["sent_len"]].values, y),
                  "auroc_length_only": auroc(y, d.text.map(n_words).values),
                  "auroc_title_start": auroc(y, [int(t.startswith(str(ti).split(" (")[0])) for t, ti in zip(d.text, d.title)]) if "title" in d.columns else np.nan,
                  "paren_rate_h": float(d[d.label == 0].text.str.contains(r"\(").mean()), "paren_rate_m": float(d[d.label == 1].text.str.contains(r"\(").mean()),
                  "median_words_h": d[d.label == 0].text.map(n_words).median(), "median_words_m": d[d.label == 1].text.map(n_words).median()})
    for c in sorted(set("".join(texts))):
        if c.isalnum() or c == " ":
            continue
        cnt = np.array([t.count(c) for t in texts])
        if cnt.sum() >= 10:
            chars.append({"set": name, "char": c, "codepoint": f"U+{ord(c):04X}", "auroc": auroc(y, cnt),
                          "df_h": float((cnt[y == 0] > 0).mean()), "df_m": float((cnt[y == 1] > 0).mean())})
audit = pd.DataFrame(audit); audit.to_csv(P["results"] / "data_audit.csv", index=False)
chars = pd.DataFrame(chars); chars["dev"] = (chars.auroc - 0.5).abs()
chars = chars.sort_values(["set", "dev"], ascending=[True, False]).groupby("set").head(15).drop(columns="dev")
chars.to_csv(P["results"] / "char_shortcuts.csv", index=False)
audit

**Report back:** `split_summary.csv`, `data_audit.csv`, `char_shortcuts.csv`, `length_matching.csv`, `meta_dropped.csv`, and the balanced N per cell.